# Tutorial 2: Circuit Discovery

This notebook teaches you how to discover circuits — the minimal sets of model components responsible for a specific behavior.

**What you'll learn:**
- Residual stream analysis with `ResidualStreamAccessor`
- Direct Logit Attribution (DLA) to find which components write the answer
- Attribution patching for fast approximate analysis
- Path patching to find connections between components
- ACDC for automated circuit discovery

**Prerequisites:** Complete [01_causal_patching.ipynb](01_causal_patching.ipynb) first.

## 1. Setup

In [ ]:
from mlxterp import InterpretableModel
from mlxterp.causal import (
    activation_patching,
    direct_logit_attribution,
    attribution_patching,
    path_patching,
    acdc,
    ResidualStreamAccessor,
)
import mlx.core as mx

model = InterpretableModel("mlx-community/Llama-3.2-1B-Instruct-4bit")
print(f"Loaded model with {len(model.layers)} layers")

## 2. The Residual Stream

Transformers process information through a **residual stream** — a hidden state that each layer reads from and writes to. Understanding the residual stream is the foundation for circuit analysis.

At each layer:
- `resid_pre` = input to the layer
- `attn_contribution` = what the attention module adds
- `resid_mid` = resid_pre + attn_contribution
- `mlp_contribution` = what the MLP adds
- `resid_post` = resid_mid + mlp_contribution = output of the layer

In [ ]:
text = "The capital of France is"

# Run a trace to capture all activations
with model.trace(text) as trace:
    output = model.output.save()

# Create a ResidualStreamAccessor
rs = ResidualStreamAccessor(trace.activations)

print(f"Available layers: {rs.available_layers()}")
print(f"Total activation keys captured: {len(trace.activations)}")

In [ ]:
# Examine the residual stream at each layer
print("Layer-by-layer residual stream analysis (last token):")
print(f"{'Layer':>5}  {'|resid_pre|':>12}  {'|attn|':>10}  {'|mlp|':>10}  {'|resid_post|':>13}")
print("-" * 60)

for layer_idx in rs.available_layers():
    pre = rs.resid_pre(layer_idx)
    attn = rs.attn_contribution(layer_idx)
    mlp = rs.mlp_contribution(layer_idx)
    post = rs.resid_post(layer_idx)
    
    # Compute norms at the last token position
    def norm(x):
        if x is None: return 0.0
        v = x[0, -1, :].astype(mx.float32)
        return float(mx.sqrt(mx.sum(v * v)))
    
    print(f"{layer_idx:>5}  {norm(pre):>12.2f}  {norm(attn):>10.2f}  {norm(mlp):>10.2f}  {norm(post):>13.2f}")

In [ ]:
# Verify the residual stream invariant: resid_pre[i] == resid_post[i-1]
print("Verifying residual chain:")
layers = rs.available_layers()
for i in range(1, min(len(layers), 5)):
    prev_post = rs.resid_post(layers[i-1])
    curr_pre = rs.resid_pre(layers[i])
    if prev_post is not None and curr_pre is not None:
        diff = float(mx.max(mx.abs(prev_post - curr_pre)))
        status = "✓" if diff < 1e-4 else "✗"
        print(f"  {status} resid_post[{layers[i-1]}] == resid_pre[{layers[i]}]  (max diff: {diff:.2e})")

## 3. Direct Logit Attribution (DLA)

DLA answers: **how much does each layer's attention and MLP contribute to predicting the final token?**

It works by projecting each component's output through the unembedding matrix to see how many "logits" it adds for the predicted token.

In [ ]:
result = direct_logit_attribution(
    model,
    text="The capital of France is",
    position=-1,  # Analyze the last token position
)

print(f"Target token: '{result.target_token_str}' (id={result.target_token})")
print(f"\n{result.summary()}")
print(f"\nAttention contributions by layer: {result.head_contributions.tolist()}")
print(f"MLP contributions by layer: {result.mlp_contributions.tolist()}")

In [ ]:
# Visualize: which layers contribute most?
import matplotlib.pyplot as plt

n_layers = len(result.head_contributions.tolist())
layers = list(range(n_layers))
attn_vals = result.head_contributions.tolist()
mlp_vals = result.mlp_contributions.tolist()

fig, ax = plt.subplots(figsize=(12, 5))
width = 0.35
ax.bar([l - width/2 for l in layers], attn_vals, width, label='Attention', color='steelblue')
ax.bar([l + width/2 for l in layers], mlp_vals, width, label='MLP', color='coral')
ax.set_xlabel('Layer')
ax.set_ylabel(f'Logit contribution for "{result.target_token_str}"')
ax.set_title('Direct Logit Attribution')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# DLA for a specific target token
rome_token = model.tokenizer.encode(" Rome")[-1]

result_rome = direct_logit_attribution(
    model,
    text="The capital of France is",
    target_token=rome_token,
)

print(f"DLA for 'Rome': {result_rome.summary()}")
print(f"Compare: which layers push TOWARD Rome vs Paris?")

## 4. Attribution Patching (Fast Approximation)

Attribution patching is ~100x faster than brute-force activation patching. It uses finite differences to approximate the gradient of the metric with respect to each activation.

Use it for quick exploration before running full activation patching.

In [ ]:
clean = "The Eiffel Tower is in"
corrupted = "The Colosseum is in"

# Fast attribution patching
attr_result = attribution_patching(
    model, clean, corrupted,
    component="resid_post",
    metric="l2",
)

print(f"Attribution patching ({attr_result.method}):")
print(f"{attr_result.summary()}")
print(f"\nScores by layer: {attr_result.attribution_scores.tolist()}")

In [ ]:
# Compare attribution patching vs brute-force activation patching
brute_result = activation_patching(
    model, clean, corrupted,
    component="resid_post",
    metric="l2",
)

print("Comparison (attribution vs brute-force):")
print(f"{'Layer':>5}  {'Attribution':>12}  {'Brute-force':>12}")
print("-" * 35)

attr_scores = attr_result.attribution_scores.tolist()
brute_scores = brute_result.effect_matrix.tolist()

for i, (a, b) in enumerate(zip(attr_scores, brute_scores)):
    print(f"{i:>5}  {a:>12.4f}  {b:>12.4f}")

In [ ]:
# Attribution for different components
for comp in ["attn", "mlp"]:
    r = attribution_patching(
        model, clean, corrupted,
        component=comp,
        metric="l2",
    )
    print(f"{comp}: {r.summary()}")

## 5. Path Patching

Activation patching tells you **which** components matter. Path patching tells you **which connections between components** matter.

It works by freezing ALL components to their clean values EXCEPT the sender, then measuring the effect. If the output changes, the sender's signal was transmitted to downstream components.

In [ ]:
# Does the attention at layer 5 send information used by later layers?
pp_result = path_patching(
    model, clean, corrupted,
    sender="layers.5.self_attn",
    receiver="layers.10.mlp",
    metric="l2",
)

print(f"Path: layers.5.self_attn → layers.10.mlp")
print(f"  Effect: {pp_result.data['effect']:.4f}")
print(f"  Components frozen: {pp_result.data['n_frozen']}")

In [ ]:
# Scan for important paths: attention → MLP connections
# (This takes a while — each path patching run freezes many components)
print("Scanning attention → MLP paths...")
print(f"{'Sender':>25} → {'Receiver':>25}  {'Effect':>8}")
print("-" * 70)

# Test a few representative paths
test_pairs = [
    (3, 8), (5, 10), (7, 12), (10, 14),
]

for sender_l, receiver_l in test_pairs:
    if sender_l >= len(model.layers) or receiver_l >= len(model.layers):
        continue
    result = path_patching(
        model, clean, corrupted,
        sender=f"layers.{sender_l}.self_attn",
        receiver=f"layers.{receiver_l}.mlp",
        metric="l2",
    )
    effect = result.data['effect']
    marker = " ← important" if abs(effect) > 0.05 else ""
    print(f"  L{sender_l}.attn → L{receiver_l}.mlp:  {effect:>8.4f}{marker}")

## 6. ACDC: Automated Circuit Discovery

ACDC automates the process: it tests every component, prunes those below a threshold, and returns the minimal circuit.

In [ ]:
circuit = acdc(
    model, clean, corrupted,
    threshold=0.01,
    components=["attn", "mlp"],
    verbose=True,
)

print(f"\n{circuit.summary()}")

In [ ]:
# Examine the discovered circuit
print("Circuit nodes (important components):")
for node in circuit.nodes:
    effect = circuit.data['node_effects'].get(node, 0)
    print(f"  {node}: effect = {effect:.4f}")

print(f"\nCircuit edges: {len(circuit.edges)}")
for sender, receiver, weight in circuit.edges[:10]:  # Show first 10
    print(f"  {sender} → {receiver} (weight={weight:.4f})")

In [ ]:
# Export as graph structure
graph = circuit.to_graph()
print(f"Graph: {len(graph['nodes'])} nodes, {len(graph['edges'])} edges")

# JSON export for external tools
import json
print(json.dumps(json.loads(circuit.to_json()), indent=2)[:500] + "...")

In [ ]:
# Try different thresholds
for threshold in [0.001, 0.01, 0.05, 0.1]:
    c = acdc(model, clean, corrupted, threshold=threshold)
    print(f"threshold={threshold:.3f}: {c.summary()}")

## 7. Putting It All Together: A Circuit Discovery Workflow

Here's the recommended workflow for discovering a circuit:

In [ ]:
# Step 1: DLA — quick overview of which layers write the answer
print("=== Step 1: Direct Logit Attribution ===")
dla = direct_logit_attribution(model, text="The capital of France is")
print(f"Target: '{dla.target_token_str}'")
print(f"Top attention layers (by contribution):")
attn_scores = list(enumerate(dla.head_contributions.tolist()))
attn_scores.sort(key=lambda x: abs(x[1]), reverse=True)
for layer, score in attn_scores[:3]:
    print(f"  Layer {layer}: {score:.4f}")

# Step 2: Activation patching — confirm with causal evidence  
print("\n=== Step 2: Activation Patching ===")
patch = activation_patching(model, clean, corrupted, component="mlp", metric="l2")
print("Top MLP layers:")
for layer, effect in patch.top_components(k=3):
    print(f"  Layer {layer}: {effect:.4f}")

# Step 3: ACDC — automated full circuit
print("\n=== Step 3: ACDC Circuit Discovery ===")
circuit = acdc(model, clean, corrupted, threshold=0.01)
print(circuit.summary())
for node in circuit.nodes:
    print(f"  {node}")

## Summary

| Technique | Speed | What It Finds | When to Use |
|-----------|-------|---------------|-------------|
| **DLA** | Very fast (1 pass) | Per-component logit contributions | First overview |
| **Attribution patching** | Fast (~2 passes) | Approximate component importance | Quick exploration |
| **Activation patching** | Medium (N+2 passes) | Exact component importance | Ground truth |
| **Path patching** | Slow (freezes many) | Connection-level importance | Specific paths |
| **ACDC** | Slow (tests all nodes) | Full minimal circuit | Comprehensive analysis |

**Next:** Try [03_generation.ipynb](03_generation.ipynb) to learn about text generation with interventions.